### 뉴스데이터가 1000갸
- 본문을 읽어서 정형 데이터로 변환하는 자동화 파이프라인을 만들어 봅시다
- 1000개 데이터의 내용을 카테고리는? 핵심어는? 요약하면? 긍정적인 내용인지 부정적인 내용인지? 
- 항목 
    - 카테고리, 
    - 요약
    - 핵심어
구조화된 출력    

### 파이프라인 설계
1. 읽기 : 뉴스 본문 불러오기
2. 분석 : 각

In [1]:
import pandas as pd 

df = pd.read_csv("../data/11-1_뉴스정제.csv")
df = df.head(5)


In [2]:
# 가장 간단한 대화 - USER 만 쓰
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()       # api 
client = OpenAI()

In [21]:
import json

# 읽기 기능
def get_data_frame_read(index):

    try:
        idx = int(index)
    except (ValueError, TypeError):
        idx = index  # 문자열 인덱스인 경우 대비

    # 실제로는 날씨 웹 검색이 들어가야 하는 거지만 , 임시로 고정값
    row = df.loc[index]
    return row.to_json(orient="records", force_ascii=False)

# 쓰기 기능 
def set_data_frame_write(summary):

    with open("summary_log.txt","a",encoding="utf-8") as f:
        f.write(summary + "\n")

        return f"성공적으로 summary_log.txt 저장 되었습니다."


tools = [{
    "type": "function",
    "function": {
        "name": "get_data_frame_read",
        "description": "데이터 프레임 의 인덱스를 받아서 해당 인덱스 에 해당하는 데이터 프레임 내용을 받는다.",
        "parameters": {
            "type": "object",
            "properties": {
                "index":    {"type": "integer", "description": "인덱스에 해당하는 데이터 프레임"},
            },
            "required": ["index"],
        },
    },
},

{
    "type": "function",
    "function": {
        "name": "set_data_frame_write",
        "description": "요약된 내용을 저장한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "summary": {"type": "string", "description": "요약된 내용"},
            },
            "required": ["summary"],
        },
    },
}
]




In [26]:
# 하나의 함수로 만들어서 사용
available_tools = {"get_data_frame_read": get_data_frame_read,"set_data_frame_write":set_data_frame_write}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content



for i in range(5):
    chat_with_tools(
        f"데이터 프레임 index {i}를 읽어서 저장해줘",
        tools
    )
    print("실행")


실행
실행
실행
실행
실행
